<a href="https://colab.research.google.com/github/Limeng-svg/Grounded-PPE-Safety-Copilot/blob/main/notebooks/01_yoloworld_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q "ultralytics==8.4.115"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 13.3 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import platform
import sys
import torch
import ultralytics

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Grounded-PPE-Safety-Copilot/phase1_outputs"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

environment = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "ultralytics": ultralytics.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "gpu": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    ),
}

print(json.dumps(environment, indent=2, ensure_ascii=False))

with open(OUTPUT_DIR / "environment.json", "w") as file:
    json.dump(environment, file, indent=2, ensure_ascii=False)

assert torch.cuda.is_available(), "请先为 Colab 启用 GPU"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
{
  "python": "3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "torch": "2.11.0+cu128",
  "ultralytics": "8.4.115",
  "cuda_available": true,
  "cuda_version": "12.8",
  "gpu": "Tesla T4"
}


In [4]:
from os import name
from google.colab import files

uploaded = files.upload()
image_paths = [Path("/content") / name for name in uploaded] # good for using for other times

print("uploaded number: ", len(image_paths))
for path in image_paths:
  print(path)

Saving ppe_easy_compliant.jpg to ppe_easy_compliant.jpg
Saving ppe_hard_crowded.jpg to ppe_hard_crowded.jpg
Saving ppe_medium_violation.jpg to ppe_medium_violation.jpg
uploaded number:  3
/content/ppe_easy_compliant.jpg
/content/ppe_hard_crowded.jpg
/content/ppe_medium_violation.jpg


In [5]:
from ultralytics import YOLOWorld

MODEL_NAME = "yolov8s-worldv2.pt"
PROMPTS = ["person", "hard hat", "safety vest"]

model = YOLOWorld(MODEL_NAME)
model.set_classes(PROMPTS)

all_results = []

for image_path in image_paths:
    result = model.predict(
        source=str(image_path),
        conf=0.20,
        iou=0.50,
        imgsz=640,
        device=0,
        verbose=False,
    )[0]

    stem = image_path.stem

    result.save(
        filename=str(OUTPUT_DIR / f"{stem}_prediction.jpg")
    )

    json_path = OUTPUT_DIR / f"{stem}_detections.json"
    json_path.write_text(
        result.to_json(),
        encoding="utf-8",
    )

    all_results.append({
        "image": image_path.name,
        "detections": len(result.boxes),
        "speed_ms": result.speed,
    })

print(json.dumps(all_results, indent=2, ensure_ascii=False))

requirements: Ultralytics requirement ['git+https://github.com/ultralytics/CLIP.git'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 37 packages in 707ms
Prepared 2 packages in 2.10s
Installed 2 packages in 1ms
 + clip==1.0 (from git+https://github.com/ultralytics/CLIP.git@68dce32140994dfcb645a1320c4ebdc034fc19fd)
 + ftfy==6.3.1

requirements: AutoUpdate success ✅ 3.2s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



100%|████████████████████████████████████████| 338M/338M [00:02<00:00, 128MiB/s]


[
  {
    "image": "ppe_easy_compliant.jpg",
    "detections": 1,
    "speed_ms": {
      "preprocess": 75.12661300006584,
      "inference": 13.80265300008432,
      "postprocess": 1.5778120000504714
    }
  },
  {
    "image": "ppe_hard_crowded.jpg",
    "detections": 17,
    "speed_ms": {
      "preprocess": 2.675693999890427,
      "inference": 49.86853500008692,
      "postprocess": 4.985050999948726
    }
  },
  {
    "image": "ppe_medium_violation.jpg",
    "detections": 1,
    "speed_ms": {
      "preprocess": 2.502397000171186,
      "inference": 47.83059500005038,
      "postprocess": 1.161751999916305
    }
  }
]


模型输入包括一张 RGB 图片和一组文本类别提示词，本实验使用 person、hard hat 和 safety vest。模型对每个预测目标输出边界框坐标、类别编号、类别名称和置信度。边界框采用 xyxy 格式，分别表示左上角和右下角坐标。降低置信度阈值通常能够发现更多目标并提高召回率，但也可能增加误检；提高阈值通常能够减少误检，但可能增加漏检。YOLO‑World 能根据文本提示进行零样本检测，但在小目标、遮挡、多人场景和领域外图片上可能出现漏检或错误类别。它只能检测目标，不能可靠判断某个头盔或背心属于哪名人员，该问题将在后续人员–PPE 关联阶段解决。